Movie Recommender (Content-Based)

This notebook builds a simple content-based movie recommender.
It uses metadata (genres, cast, director, overview text) to build a TF-IDF vector for each film, then recommends movies using cosine similarity to a user’s four favorite titles.

In [131]:
import pandas as pd
from ipywidgets import Dropdown, Button, VBox, HBox, Output, Label
from collections import Counter
from IPython.display import display, Markdown
import numpy as np
import re
import ast
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

movies = pd.read_csv("tmdb_5000_movies.csv")
credits = pd.read_csv("tmdb_5000_credits.csv")


In [132]:
# normalize ratings globally to 0–1
r_min = movies["vote_average"].min()
r_max = movies["vote_average"].max()
movies["rating_norm"] = (movies["vote_average"] - r_min) / (r_max - r_min)


In [133]:
credits.head()

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [134]:
movies.shape, credits.shape

((4803, 21), (4803, 4))

In [135]:
# we want to merge the two CSVs with movies.id and credits.movie_id
movies.columns, credits.columns

(Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
        'original_title', 'overview', 'popularity', 'production_companies',
        'production_countries', 'release_date', 'revenue', 'runtime',
        'spoken_languages', 'status', 'tagline', 'title', 'vote_average',
        'vote_count', 'rating_norm'],
       dtype='object'),
 Index(['movie_id', 'title', 'cast', 'crew'], dtype='object'))

In [136]:
movies = movies.merge(credits, left_on="id", right_on="movie_id")
movies

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,...,status,tagline,title_x,vote_average,vote_count,rating_norm,movie_id,title_y,cast,crew
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...",...,Released,Enter the World of Pandora.,Avatar,7.2,11800,0.72,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...",...,Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,0.69,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...",...,Released,A Plan No One Escapes,Spectre,6.3,4466,0.63,206647,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...",...,Released,The Legend Ends,The Dark Knight Rises,7.6,9106,0.76,49026,The Dark Knight Rises,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]",...,Released,"Lost in our world, found in another.",John Carter,6.1,2124,0.61,49529,John Carter,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4798,220000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",NaN,9367,"[{""id"": 5616, ""name"": ""united states\u2013mexi...",es,El Mariachi,El Mariachi just wants to play his guitar and ...,14.269792,"[{""name"": ""Columbia Pictures"", ""id"": 5}]",...,Released,"He didn't come looking for trouble, but troubl...",El Mariachi,6.6,238,0.66,9367,El Mariachi,"[{""cast_id"": 1, ""character"": ""El Mariachi"", ""c...","[{""credit_id"": ""52fe44eec3a36847f80b280b"", ""de..."
4799,9000,"[{""id"": 35, ""name"": ""Comedy""}, {""id"": 10749, ""...",NaN,72766,[],en,Newlyweds,A newlywed couple's honeymoon is upended by th...,0.642552,[],...,Released,A newlywed couple's honeymoon is upended by th...,Newlyweds,5.9,5,0.59,72766,Newlyweds,"[{""cast_id"": 1, ""character"": ""Buzzy"", ""credit_...","[{""credit_id"": ""52fe487dc3a368484e0fb013"", ""de..."
4800,0,"[{""id"": 35, ""name"": ""Comedy""}, {""id"": 18, ""nam...",http://www.hallmarkchannel.com/signedsealeddel...,231617,"[{""id"": 248, ""name"": ""date""}, {""id"": 699, ""nam...",en,

In [137]:
movies.columns, movies.shape
# movies df now with 4 extra columns: movie_id, title_y, cast, crew (title_x, title_y added due to being duplicate title)

(Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
        'original_title', 'overview', 'popularity', 'production_companies',
        'production_countries', 'release_date', 'revenue', 'runtime',
        'spoken_languages', 'status', 'tagline', 'title_x', 'vote_average',
        'vote_count', 'rating_norm', 'movie_id', 'title_y', 'cast', 'crew'],
       dtype='object'),
 (4803, 25))

In [138]:
movies["title"] = movies["title_x"]
movies = movies.drop(columns = ['title_x', 'title_y'])

In [139]:
movies[["genres", "keywords"]]
# must parse columns like these to readible strings 

,genres,keywords
0,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":..."
1,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na..."
2,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name..."
3,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...","[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,..."
4,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 818, ""name"": ""based on novel""}, {""id"":..."
...,...,...
4798,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...","[{""id"": 5616, ""name"": ""united states\u2013mexi..."
4799,"[{""id"": 35, ""name"": ""Comedy""}, {""id"": 10749, ""...",[]
4800,"[{""id"": 35, ""name"": ""Comedy""}, {""id"": 18, ""nam...","[{""id"": 248, ""name"": ""date""}, {""id"": 699, ""nam..."
4801,[],[]


In [140]:
def parse_names(x, key="name"):
    try:
        data = ast.literal_eval(x)  # turn string into real Python list/dicts
    except (ValueError, SyntaxError, TypeError):
        return []
    names = []
    for d in data:
        if isinstance(d, dict) and key in d:
            names.append(d[key].replace(" ", "_").lower())
    return names

In [141]:
movies["genres_clean"]   = movies["genres"].apply(parse_names)
movies["keywords_clean"] = movies["keywords"].apply(parse_names)


In [142]:
movies["cast_clean"] = movies["cast"].apply(lambda x: parse_names(x)[:5])

In [143]:
movies["cast_clean"]

0       [sam_worthington, zoe_saldana, sigourney_weave...
1       [johnny_depp, orlando_bloom, keira_knightley, ...
2       [daniel_craig, christoph_waltz, léa_seydoux, r...
3       [christian_bale, michael_caine, gary_oldman, a...
4       [taylor_kitsch, lynn_collins, samantha_morton,...
                              ...                        
4798    [carlos_gallardo, jaime_de_hoyos, peter_marqua...
4799    [edward_burns, kerry_bishé, marsha_dietlein, c...
4800    [eric_mabius, kristin_booth, crystal_lowe, geo...
4801    [daniel_henney, eliza_coupe, bill_paxton, alan...
4802    [drew_barrymore, brian_herzlinger, corey_feldm...
Name: cast_clean, Length: 4803, dtype: object

In [144]:
def get_director(crew_str):
    try:
        crew = ast.literal_eval(crew_str)
    except (ValueError, SyntaxError, TypeError):
        return []
    for d in crew:
        if isinstance(d, dict) and d.get("job") == "Director":
            return [d["name"].replace(" ", "_").lower()]
    return []

movies["director_clean"] = movies["crew"].apply(get_director)

In [145]:
movies[["title", "genres_clean", "keywords_clean", "cast_clean", "director_clean"]].head()


,title,genres_clean,keywords_clean,cast_clean,director_clean
0,Avatar,"[action, adventure, fantasy, science_fiction]","[culture_clash, future, space_war, space_colon...","[sam_worthington, zoe_saldana, sigourney_weave...",[james_cameron]
1,Pirates of the Caribbean: At World's End,"[adventure, fantasy, action]","[ocean, drug_abuse, exotic_island, east_india_...","[johnny_depp, orlando_bloom, keira_knightley, ...",[gore_verbinski]
2,Spectre,"[action, adventure, crime]","[spy, based_on_novel, secret_agent, sequel, mi...","[daniel_craig, christoph_waltz, léa_seydoux, r...",[sam_mendes]
3,The Dark Knight Rises,"[action, crime, drama, thriller]","[dc_comics, crime_fighter, terrorist, secret_i...","[christian_bale, michael_caine, gary_oldman, a...",[christopher_nolan]
4,John Carter,"[action, adventure, science_fiction]","[based_on_novel, mars, medallion, space_travel...","[taylor_kitsch, lynn_collins, samantha_morton,...",[andrew_stanton]


In [146]:
movies["overview_clean"] = movies["overview"].fillna("").str.lower()

def make_soup(row):
    genres    = row["genres_clean"]
    keywords  = row["keywords_clean"]
    cast      = row["cast_clean"]
    director  = row["director_clean"]
    overview  = row["overview_clean"].split()[:50]  # first 50 words

    return " ".join(genres + keywords + cast + director + overview)

movies["soup"] = movies.apply(make_soup, axis=1)

In [147]:
tfidf_vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf_vectorizer.fit_transform(movies["soup"])
sim_tfidf = cosine_similarity(tfidf_matrix, tfidf_matrix)

count_vectorizer = CountVectorizer(stop_words="english")
count_matrix = count_vectorizer.fit_transform(movies["soup"])
sim_counts = cosine_similarity(count_matrix, count_matrix)

# Blend them: alpha controls how much you trust TF-IDF vs counts
alpha = 0.7  #
sim_matrix = alpha * sim_tfidf + (1 - alpha) * sim_counts


#each dimension is TF-IDF = term frequency × inverse document frequency

- We represent each movie as a TF-IDF vector over words in its metadata and overview.
- The similarity between two movies is computed with cosine similarity (the cosine of the angle between their vectors).
- A higher cosine value means the movies share more features (genres, cast, plot keywords, etc.).

In [148]:
indices = pd.Series(movies.index, index=movies["title"].str.lower()).drop_duplicates()


In [149]:
def build_favorite_sets(fav_idxs):
    fav_genres = set()
    fav_cast = set()
    fav_directors = set()
    for idx in fav_idxs:
        row = movies.iloc[idx]
        fav_genres.update(row["genres_clean"])
        fav_cast.update(row["cast_clean"])
        fav_directors.update(row["director_clean"])
    return fav_genres, fav_cast, fav_directors

def intersect_sorted(a_list, fav_set):
    return sorted(set(a_list) & fav_set)

def pretty_tokens(tokens):
    """Turn ['ethan_hawke', 'julie_delpy'] -> 'Ethan Hawke, Julie Delpy'."""
    return ", ".join(t.replace("_", " ").title() for t in tokens)

def one_sentence(text):
    """Return the first sentence of an overview."""
    text = (text or "").strip()
    if not text:
        return ""
    parts = re.split(r'(?<=[.!?])\s+', text)
    return parts[0].strip()

def summarize_match_reason(shared_genres, shared_cast, shared_director):
    reasons = []
    if shared_director:
        reasons.append(f"same director as one of your favorites ({pretty_tokens(shared_director)})")
    if shared_cast:
        reasons.append(f"overlapping cast ({pretty_tokens(shared_cast[:3])})")
    if shared_genres:
        reasons.append(f"similar genres ({pretty_tokens(shared_genres[:3])})")
    if not reasons:
        reasons.append("overall similarity in themes and metadata")
    return "; ".join(reasons)

def dominant_genres_from_favs(fav_idxs, top_k=2):
    """Count genre frequency across favorites and return top_k genre names."""
    all_genres = []
    for idx in fav_idxs:
        all_genres.extend(movies.iloc[idx]["genres_clean"])
    counts = Counter(all_genres)
    if not counts:
        return []
    return [g for g, _ in counts.most_common(top_k)]


def summarize_and_recommend_for_indices(fav_idxs, n_recs=10, min_match=0.5):
    fav_genres, fav_cast, fav_directors = build_favorite_sets(fav_idxs)
    dom_genres = dominant_genres_from_favs(fav_idxs, top_k=2) 
    
    agg_sim = np.zeros(sim_matrix.shape[0])
    for idx in fav_idxs:
        agg_sim += sim_matrix[idx]
    
    for idx in fav_idxs:
        agg_sim[idx] = -1
    
    candidate_mask = movies["genres_clean"].apply(
        lambda gs: any(g in gs for g in dom_genres)
    )
    candidate_indices = np.where(candidate_mask)[0]
    if len(candidate_indices) == 0:
        candidate_indices = agg_sim.argsort()[::-1][:n_recs * 5]
    
    candidate_indices = candidate_indices[np.argsort(agg_sim[candidate_indices])[::-1]]
    candidate_indices = candidate_indices[: n_recs * 5]  # oversample a bit
    recs = movies.iloc[candidate_indices].copy()
    
    scores = agg_sim[candidate_indices]
    if scores.max() > scores.min():
        norm_scores = (scores - scores.min()) / (scores.max() - scores.min())
    else:
        norm_scores = np.ones_like(scores)
    recs["raw_match"] = norm_scores  # keep this for filtering & later rescale
    
    
    beta = 0.2
    recs["rating_norm"] = recs["rating_norm"].fillna(0.5)  # fallback if any NaNs
    recs["blended_score"] = (1 - beta) * recs["raw_match"] + beta * recs["rating_norm"]
    
    recs["shared_genres"] = recs["genres_clean"].apply(lambda g: intersect_sorted(g, fav_genres))
    recs["shared_cast"] = recs["cast_clean"].apply(lambda c: intersect_sorted(c, fav_cast))
    recs["shared_director"] = recs["director_clean"].apply(lambda d: intersect_sorted(d, fav_directors))
    recs["overview_short"] = recs["overview"].fillna("").apply(one_sentence)
    
    filtered = recs[recs["blended_score"] >= min_match]
    
    target_k = max(3, n_recs)
    if len(filtered) >= target_k:
        recs_final = filtered.sort_values("blended_score", ascending=False).head(n_recs)
    else:
        recs_final = recs.sort_values("blended_score", ascending=False).head(target_k)
    
    sel_scores = recs_final["blended_score"].to_numpy()
    if sel_scores.max() > sel_scores.min():
        renorm = (sel_scores - sel_scores.min()) / (sel_scores.max() - sel_scores.min())
    else:
        renorm = np.ones_like(sel_scores)
    recs_final["match_percent"] = (50 + 50 * renorm).round(1)  # 50–100
    
    recs_final.attrs["dominant_genres"] = dom_genres
    return recs_final




In [150]:
titles = sorted(movies["title"].unique())
title_options = ["-- choose a movie --"] + titles

fav1 = Dropdown(options=title_options, description="Favorite 1")
fav2 = Dropdown(options=title_options, description="Favorite 2")
fav3 = Dropdown(options=title_options, description="Favorite 3")
fav4 = Dropdown(options=title_options, description="Favorite 4")

run_button = Button(description="Recommend", button_style="primary")
output = Output()

def on_run_clicked(b):
    output.clear_output()
    fav_titles = [fav1.value, fav2.value, fav3.value, fav4.value]
    
    with output:
        if any(t == "-- choose a movie --" for t in fav_titles):
            display(Markdown("⚠️ **Please choose 4 favorite movies (no placeholders).**"))
            return
        
        fav_idxs = []
        missing = []
        for t in fav_titles:
            key = t.lower()
            if key in indices:
                fav_idxs.append(indices[key])
            else:
                missing.append(t)
        
        if missing:
            display(Markdown(f"⚠️ These titles were not found in the dataset: {', '.join(missing)}"))
            return
        
        display(Markdown("### 🎬 Your favorites"))
        for idx in fav_idxs:
            row = movies.iloc[idx]
            genres = pretty_tokens(row["genres_clean"])
            cast = pretty_tokens(row["cast_clean"])
            director = pretty_tokens(row["director_clean"])
            rating = row.get("vote_average", None)
            votes = row.get("vote_count", None)

            rating_str = ""
            if rating is not None and votes is not None:
                rating_str = f"*Rating:* {rating:.1f}/10 (based on {int(votes)} votes)  \n"

            display(Markdown(
                f"**{row['title']}**  \n"
                f"*Genres:* {genres}  \n"
                f"*Cast:* {cast}  \n"
                f"*Director:* {director}  \n"
                f"{rating_str}"
            ))
            display(Markdown("---"))
        
        # recommendations
        recs = summarize_and_recommend_for_indices(fav_idxs, n_recs=5, min_match=0.5)

        dom_genres = recs.attrs.get("dominant_genres", [])
        if dom_genres:
            display(Markdown(
                f"**Detected dominant genres from your favorites:** {pretty_tokens(dom_genres)}"
            ))

        display(Markdown("### 🎥 Recommendations (match score rescaled to 50–100, blended with rating)"))
        
        for _, row in recs.iterrows():
            shared_genres = row["shared_genres"]
            shared_cast = row["shared_cast"]
            shared_director = row["shared_director"]
            reason = summarize_match_reason(shared_genres, shared_cast, shared_director)
            
            genres = pretty_tokens(row["genres_clean"])
            cast = pretty_tokens(row["cast_clean"])
            director = pretty_tokens(row["director_clean"])
            rating = row.get("vote_average", None)
            votes = row.get("vote_count", None)

            rating_str = ""
            if rating is not None and votes is not None:
                rating_str = f"*Rating:* {rating:.1f}/10 (based on {int(votes)} votes)  \n"

            display(Markdown(
                f"**{row['title']}** — **match: {row['match_percent']}%**  \n"
                f"*Reason:* {reason}  \n"
                f"*Genres:* {genres}  \n"
                f"*Cast:* {cast}  \n"
                f"*Director:* {director}  \n"
                f"{rating_str}\n"
                f"*Overview:* {row['overview_short']}"
            ))
            display(Markdown("---"))

run_button.on_click(on_run_clicked)

ui = VBox([
    Label("Pick exactly 4 favorite movies from this dataset (mostly films released up to ~2015). You can type to search in each dropdown:"),
    HBox([fav1, fav2]),
    HBox([fav3, fav4]),
    run_button,
    output
])

display(ui)
